In [5]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from logistic_models import sigmoid, fit_logistic, compute_confusion_matrix

In [6]:
rng = np.random.default_rng(seed=42)
n_samples = 100
n_features = 2

X = rng.standard_normal((n_samples, n_features))
X_b = np.hstack((np.ones((n_samples, 1)), X))

true_beta = np.array([[0.5], [2.5], [-1.5]])
probabilities = sigmoid(X_b @ true_beta)
Y = (probabilities > 0.5).astype(int)

In [7]:
beta_optimise, historique = fit_logistic(X_b, Y, lr=0.1, num_iter=1000)

print("--- SUIVI DE L'ENTRAÎNEMENT ---")
for iter_idx, loss_val in historique:
    print(f"Itération {iter_idx} -> Erreur (Loss): {loss_val:.4f}")

print("\nBetas finaux optimisés :\n", beta_optimise)

--- SUIVI DE L'ENTRAÎNEMENT ---
Itération 0 -> Erreur (Loss): 0.6931
Itération 200 -> Erreur (Loss): 0.2341
Itération 400 -> Erreur (Loss): 0.1751
Itération 600 -> Erreur (Loss): 0.1482
Itération 800 -> Erreur (Loss): 0.1319

Betas finaux optimisés :
 [[ 1.19466402]
 [ 4.7286477 ]
 [-2.68320309]]


In [8]:
# Prédictions finales du modèle
Y_prob = sigmoid(X_b @ beta_optimise)
Y_pred_class = (Y_prob >= 0.5).astype(int)

# Calcul des indicateurs de performance
TN, FP, FN, TP = compute_confusion_matrix(Y, Y_pred_class)
accuracy = (TP + TN) / len(Y)

print("=== RÉSULTATS DE LA CLASSIFICATION ===")
print(f"Accuracy : {accuracy * 100:.2f}%\n")
print("Matrice de Confusion :")
print("               Prédit 0   Prédit 1")
print(f"Réellement 0 |    {TN}     {FP}")
print(f"Réellement 1 |    {FN}     {TP}")

=== RÉSULTATS DE LA CLASSIFICATION ===
Accuracy : 98.00%

Matrice de Confusion :
               Prédit 0   Prédit 1
Réellement 0 |    38     2
Réellement 1 |    0     60


In [ ]:
" Remarques "

# 1. A quoi sert .ravel() ?
# --> s'applique à Y_pred qui est une matrice colonnne
# --> "aplatit" Y
# --> on passe d'une vecteur colonne à une liste 

# exemple : 
# Y= [[1]]
#     [0]
#     [0]]
# --> ravel donne : [1,0,0] (liste 1D)

# --> plus facile de comparer terme à terme des listes que des vecteurs colonnes


# 2. A quoi sert astype(int)
# --> Convertit les True/False en 1/0


# 3. Pourquoi l'erreur démarre précisément à 0.6931 ?
# --> À l'itération 0, l'erreur vaut 0.6931
# --> On a initialisé tous nos coefficients zéro
# --> Notre modèle n'en sait absolument rien et prédit une probabilité parfaite de 0.5 (pile ou face) pour chaque échantillon
# --> sigma(0) = 0.5
# --> On calcule la fonction de coût (Log Loss) pour une prédiction constante de 0.5 
# --> La formule simplifiée donne -ln(0.5) = 0.693147
# --> C'est le comportement idéal : le modèle part d'une incertitude totale (le hasard pur), puis l'erreur chute drastiquement (de 0.69 à 0.12)
# --> Prouve que le modèle apprend efficacement !


# 4. Pourquoi les estimations des coefficients ne sont pas parfaites ?
# --> Maximiser la certitude de ses prédictions (rapprocher les probabilités au plus près de 1 ou de 0)
# --> Régression logistique cherche à rendre la pente de sa courbe sigmoïde la plus raide possible. 
# --> Pour y arriver, elle fait grimper la taille de ses coefficients beta